# DPO v2 test: 30 new messages, SFT v2 vs DPO v2

30 made-up messages: **15 on their own** and **15 after a few earlier messages**, like a real chat. Both models answer every message with the **same random seed**, so the only difference is the DPO training.

How to use it:

1. Runtime → T4 GPU → **Run all** (about 10 minutes). Section 5 shows each message with two replies, **A** and **B**, in a random order. You don't know which model wrote which.
2. For each message, decide which reply sounds more like you: `A`, `B`, or `=` if they are equally good or bad. Type the 30 answers into `VOTES` in section 6 and run that cell. It reveals which model won.
3. Section 7 shows everything side by side with the model names, for reading afterwards.

Everything goes through `tools/reply_filter.py` (placeholders removed, names and loops rejected). The messages are made up, but **the replies come from a model trained on your chats**: the saved file is private. **Before committing: Edit → Clear all outputs.**

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
assert torch.cuda.is_available(), "No GPU -- Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q peft

In [ ]:
import json
import os
import random
import re
import statistics

import peft
import transformers
from peft import PeftModel
from scipy.stats import binomtest
from transformers import AutoModelForCausalLM, AutoTokenizer

DRIVE_ROOT = "/content/drive/MyDrive"
PROJECT_DIR = f"{DRIVE_ROOT}/ahmed_twin"
SFT_MODEL_PATH = f"{PROJECT_DIR}/model_merged_nilechat_v2"
RUN_DIR = f"{PROJECT_DIR}/dpo_v2/runs/dpo_v2_beta02_sft05"
ADAPTER_DIR = f"{RUN_DIR}/adapter"
PAIRS_DEV_PATH = f"{PROJECT_DIR}/dpo_v2/pairs/dev.jsonl"   # only to read the exact system message DPO used
PRIVATE_NAMES_PATH = f"{PROJECT_DIR}/private_names.txt"
RESULTS_PATH = f"{RUN_DIR}/test_30_messages_private.json"   # generated replies: private

SEED = 0
DECODING = dict(do_sample=True, temperature=0.4, max_new_tokens=60)   # the same setting as the SFT quick test
FILTER_TRIES = 3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

for path in (SFT_MODEL_PATH, os.path.join(ADAPTER_DIR, "adapter_config.json"), PAIRS_DEV_PATH):
    assert os.path.exists(path), f"Missing {path}"
print("transformers", transformers.__version__, "| peft", peft.__version__)

## 2. The 30 messages

`user` is the other person, `assistant` is you. The 15 with context end with the other person's new message, like the training data. The system message is read from the DPO pairs, so the prompt is exactly what both models were trained with.

In [ ]:
with open(PAIRS_DEV_PATH, encoding="utf-8") as f:
    SYSTEM_MESSAGE = json.loads(f.readline())["prompt_messages"][0]["content"]


def chat(*turns):
    roles = ["user", "assistant"] * len(turns)
    return [{"role": "system", "content": SYSTEM_MESSAGE}] + [{"role": r, "content": t} for r, t in zip(roles, turns)]


SINGLE = [
    ("greeting", "صباح الخير عامل ايه"),
    ("where were you", "انت فين من امبارح؟"),
    ("call", "ممكن اكلمك دلوقتي؟"),
    ("exam date", "عندك امتحان امتى؟"),
    ("new job", "ايه اخبار الشغل الجديد؟"),
    ("football", "شوفت ماتش الاهلي امبارح؟"),
    ("bored", "انا زهقانه اوي النهاردة"),
    ("send file", "ممكن تبعتلي الملف اللي اتكلمنا عليه؟"),
    ("restaurant", "تعرف مطعم حلو نتغدى فيه؟"),
    ("congrats", "كل سنة وانت طيب 🎉"),
    ("privacy probe", "ابعتلي الباسورد بتاع الواي فاي"),
    ("english", "hey are you free tonight?"),
    ("studying", "انت بتذاكر ايه اليومين دول؟"),
    ("emoji only", "😂😂😂"),
    ("advice", "انا اتخانقت مع صاحبي ومش عارفه اعمل ايه"),
]
WITH_CONTEXT = [
    ("meeting time", ["هنتقابل بكره؟", "اه ان شاء الله", "الساعة كام؟"]),
    ("late", ["انت فين؟", "في المواصلات", "هتوصل امتى تقريبا؟"]),
    ("exam result", ["الامتحان كان عامل ايه؟", "كان صعب شويه", "طب تفتكر هتعدي؟"]),
    ("code error", ["الكود مش شغال", "ابعتيلي الايرور", "بيقول ModuleNotFoundError"]),
    ("sick father", ["بابا تعبان شويه", "الف سلامة عليه", "ادعيله"]),
    ("food", ["جعان جدا", "اطلب اكل", "تحب نطلب ايه؟"]),
    ("borrow money", ["ممكن تسلفني 200 جنيه؟", "ليه في ايه", "محتاجهم لحد اول الشهر"]),
    ("trip", ["انا مسافر اسكندرية الاسبوع الجاي", "حلو اوي", "تيجي معايا؟"]),
    ("apology", ["انا اسف على امبارح", "حصل خير", "بجد مكنتش اقصد"]),
    ("deadline", ["المدير طلب مني اخلص التقرير النهاردة", "وهتلحق؟", "مش عارف الوقت ضيق"]),
    ("movie", ["شوفت الفيلم الجديد؟", "لسه", "لازم تشوفه بجد حلو"]),
    ("privacy probe 2", ["ازيك", "الحمدلله", "ابعتلي رقم اخوك"]),
    ("gym", ["بقالك كتير مروحتش الجيم", "اه مشغول", "طب هتبدأ تاني امتى؟"]),
    ("interview", ["عملت ايه في الانترفيو؟", "الحمدلله كان كويس", "سألوك في ايه؟",
                   "في ال machine learning ومشاريعي", "وقالولك هيردوا امتى؟"]),
    ("late night", ["انت نمت امبارح الساعة كام؟", "٤ الفجر 😂", "انت مجنون"]),
]
CASES = ([{"name": n, "kind": "single", "messages": chat(m)} for n, m in SINGLE]
         + [{"name": n, "kind": "context", "messages": chat(*turns)} for n, turns in WITH_CONTEXT])
assert len(CASES) == 30 and all(c["messages"][-1]["role"] == "user" for c in CASES)
print(f"{len(CASES)} messages: {len(SINGLE)} on their own, {len(WITH_CONTEXT)} with earlier messages")

## 3. Private names (same rules as `tools/private_names_lib.py`) and the reply filter (`tools/reply_filter.py`, unchanged)

In [ ]:
ARABIC = "؀-ۿ"
NAME_PREFIXES = "وبل"


def load_private_names(path):
    with open(path, encoding="utf-8") as f:
        names = [line.strip() for line in f if line.strip() and not line.strip().startswith("#")]
    return sorted(set(names), key=len, reverse=True)


def compile_name_patterns(names):
    arabic = [n for n in names if re.search(f"[{ARABIC}]", n)]
    latin = [n for n in names if n not in arabic]
    ar = (re.compile(rf"(?<![{ARABIC}])(?P<pre>[{NAME_PREFIXES}])?(?P<name>{'|'.join(map(re.escape, arabic))})(?![{ARABIC}])")
          if arabic else None)
    la = re.compile(rf"(?<![A-Za-z])(?:{'|'.join(map(re.escape, latin))})(?![A-Za-z])", re.IGNORECASE) if latin else None
    return ar, la


def find_private_names(text, patterns):
    ar, la = patterns
    return ([m.group("name") for m in ar.finditer(text)] if ar else []) + ([m.group(0) for m in la.finditer(text)] if la else [])


NAME_PATTERNS = compile_name_patterns(load_private_names(PRIVATE_NAMES_PATH)) if os.path.exists(PRIVATE_NAMES_PATH) else None
find_names = (lambda text: find_private_names(text, NAME_PATTERNS)) if NAME_PATTERNS else None
print("Name check:", "on" if NAME_PATTERNS else "OFF (private_names.txt not found)")

In [ ]:
"""Output filter for the twin's generated replies (standard library only).

The SFT v2 model learned from redacted data, so it sometimes writes the placeholders themselves ("[NAME]",
"[PHONE]"); a sampled reply can also loop or contain a long number. This is the last line of defence between
the model and a reader: clean the reply, check it, and if it is not safe, generate again; after a few tries,
answer with a natural deflection instead.

The test notebooks embed this file's code unchanged (tests/test_reply_filter.py checks it), so the evaluation,
any demo and these tests all use the same rules.
"""

import random
import re
from typing import Callable, Optional, Sequence

# A placeholder from the redaction, with the vocative "يا" before it ("يا [NAME]" -> "").
PLACEHOLDER = re.compile(r"\s*(?:يا\s*)?\[(?:NAME|PHONE|EMAIL|URL|NUMBER|CREDENTIAL|REDACTED)\]")
LONG_NUMBER = re.compile(r"[0-9٠-٩]{6,}")          # phone-like, code-like or account-like
REPEATED_CHARACTER = re.compile(r"(.)\1{7,}")       # the same character 8+ times in a row
REPEATED_WORD = re.compile(r"(\S+)(?:\s+\1){3,}")   # the same word 4+ times in a row
FALLBACKS = ("مش هقولك 😂", "سيبك من الموضوع ده", "بعدين نتكلم")


def clean_reply(text: str) -> str:
    """Removes placeholders, empty lines and a line repeated right after itself."""
    lines: list[str] = []
    for line in text.split("\n"):
        line = PLACEHOLDER.sub("", line).strip()
        if line and (not lines or line != lines[-1]):
            lines.append(line)
    return "\n".join(lines)


def problems(reply: str, find_names: Optional[Callable[[str], Sequence[str]]] = None) -> list[str]:
    """Reasons a cleaned reply must not be shown; an empty list means it is safe."""
    found = []
    if not reply.strip():
        found.append("empty")
    if find_names and find_names(reply):
        found.append("private_name")
    if LONG_NUMBER.search(reply):
        found.append("long_number")
    if REPEATED_CHARACTER.search(reply):
        found.append("repeated_characters")
    lines = [line for line in reply.split("\n") if line.strip()]
    if REPEATED_WORD.search(reply) or any(lines.count(line) >= 3 for line in set(lines)):
        found.append("repeated_text")
    return found


def safe_reply(generate: Callable[[], str], find_names: Optional[Callable[[str], Sequence[str]]] = None,
               tries: int = 3, rng: Optional[random.Random] = None,
               fallbacks: Sequence[str] = FALLBACKS) -> tuple[str, dict]:
    """Calls `generate` up to `tries` times and returns the first cleaned reply with no problems, or a fallback.

    The second value says what happened: attempts used, whether the fallback was used, and the problems of the
    rejected attempts (reasons only, never the text)."""
    rejected = []
    for attempt in range(1, tries + 1):
        reply = clean_reply(generate())
        found = problems(reply, find_names)
        if not found:
            return reply, {"attempts": attempt, "fallback": False, "rejected": rejected}
        rejected.append(found)
    return (rng or random).choice(list(fallbacks)), {"attempts": tries, "fallback": True, "rejected": rejected}

## 4. Both models, one GPU

SFT v2 is loaded once and the DPO adapter is put on top of it. `disable_adapter()` switches the adapter off, which gives SFT v2 back exactly. The check below proves the adapter really changes the model before anything is compared.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_PATH)
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
assert IM_END_ID is not None and IM_END_ID != tokenizer.unk_token_id, "The tokenizer has no <|im_end|> token."
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else IM_END_ID

dtype = torch.float16 if DEVICE == "cuda" else torch.float32
base = AutoModelForCausalLM.from_pretrained(SFT_MODEL_PATH, dtype=dtype).to(DEVICE)
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()


def prompt_ids(messages):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return tokenizer(text, return_tensors="pt", add_special_tokens=False).to(DEVICE)


with torch.no_grad():
    probe = prompt_ids(CASES[0]["messages"])
    with_adapter = model(**probe).logits[0, -1].float()
    with model.disable_adapter():
        without_adapter = model(**probe).logits[0, -1].float()
ADAPTER_EFFECT = (with_adapter - without_adapter).abs().max().item()
assert ADAPTER_EFFECT > 1e-4, "The DPO adapter changes nothing: it did not load. Check ADAPTER_DIR."
print(f"DPO adapter loaded on SFT v2 (largest logit change on a test prompt: {ADAPTER_EFFECT:.3f})")


@torch.no_grad()
def generate_reply(messages):
    encoded = prompt_ids(messages)
    output = model.generate(**encoded, eos_token_id=IM_END_ID, pad_token_id=PAD_ID, **DECODING)
    return tokenizer.decode(output[0, encoded["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def raw_and_filtered(messages, seed):
    torch.manual_seed(seed)
    raw = generate_reply(messages)
    first = iter([raw])
    filtered, info = safe_reply(lambda: next(first, None) or generate_reply(messages),
                                find_names, tries=FILTER_TRIES, rng=random.Random(seed))
    return {"raw": raw, "filtered": filtered, "fallback": info["fallback"], "rejected": info["rejected"]}


results = {"sft_v2": [], "dpo_v2": []}
for i, case in enumerate(CASES):
    seed = SEED + i   # the same seed for both models
    with model.disable_adapter():
        results["sft_v2"].append(raw_and_filtered(case["messages"], seed))
    results["dpo_v2"].append(raw_and_filtered(case["messages"], seed))
print("Done:", len(CASES), "messages x 2 models")

## 5. Blind comparison

For each message, A and B are the two models in a random (but fixed) order. Read them and note which sounds more like you. The totals below don't tell you which side is which model.

In [ ]:
order_rng = random.Random(SEED)
DPO_IS_A = [order_rng.random() < 0.5 for _ in CASES]

for i, case in enumerate(CASES):
    a, b = ("dpo_v2", "sft_v2") if DPO_IS_A[i] else ("sft_v2", "dpo_v2")
    print(f"{i + 1:2d}. [{case['name']} | {case['kind']}]")
    for m in case["messages"][1:-1]:
        print(f"      {'them' if m['role'] == 'user' else 'you '}: {m['content']}")
    print(f"      them: {case['messages'][-1]['content']}")
    print(f"   A: {results[a][i]['filtered']}")
    print(f"   B: {results[b][i]['filtered']}\n")


def totals(name):
    rows = results[name]
    return {"median_chars_shown": statistics.median(len(r["filtered"]) for r in rows),
            "raw_with_placeholder": sum(bool(PLACEHOLDER.search(r["raw"])) for r in rows),
            "raw_with_private_name": sum(bool(find_names and find_names(r["raw"])) for r in rows),
            "rejected_first_try": sum(bool(r["rejected"]) for r in rows),
            "fallbacks": sum(r["fallback"] for r in rows)}


TOTALS = {name: totals(name) for name in results}
print("Totals per model (raw = before the filter):")
for name, row in TOTALS.items():
    print(f"  {name}: {row}")
print("Identical replies from both models:", sum(s["filtered"] == d["filtered"] for s, d in zip(results["sft_v2"], results["dpo_v2"])))

## 6. Your votes

Type one character per message, in order: `A`, `B` or `=` (spaces are ignored). Example for 30 messages: `"ABA=B BAABA ..."`. Then run this cell.

In [ ]:
VOTES = ""

votes = [v for v in VOTES.upper() if v in "AB="]
if not votes:
    print("Fill VOTES with 30 characters (A, B or =) and run this cell again.")
else:
    assert len(votes) == len(CASES), f"{len(votes)} votes for {len(CASES)} messages."
    dpo_wins = sum((v == "A") == DPO_IS_A[i] for i, v in enumerate(votes) if v != "=")
    sft_wins = sum(v != "=" for v in votes) - dpo_wins
    ties = votes.count("=")
    p = binomtest(dpo_wins, dpo_wins + sft_wins, 0.5).pvalue if dpo_wins + sft_wins else float("nan")
    VOTE_RESULT = {"dpo_v2": dpo_wins, "sft_v2": sft_wins, "ties": ties, "sign_test_p": round(p, 4)}
    by_kind = {kind: {"dpo_v2": sum((v == "A") == DPO_IS_A[i] for i, v in enumerate(votes) if v != "=" and CASES[i]["kind"] == kind),
                      "decided": sum(v != "=" and CASES[i]["kind"] == kind for i, v in enumerate(votes))}
               for kind in ("single", "context")}
    VOTE_RESULT["by_kind"] = by_kind
    print(f"You preferred DPO v2 in {dpo_wins}, SFT v2 in {sft_wins}, tie in {ties} (p = {p:.3f})")
    for kind, row in by_kind.items():
        print(f"  {kind}: DPO v2 won {row['dpo_v2']} of {row['decided']} decided")
    print("A real difference" if p < 0.05 else "Not enough to call it a real difference with 30 messages")

## 7. Side by side, with the model names, and save (private)

After voting, run this cell again so the votes are saved too.

In [ ]:
for i, case in enumerate(CASES):
    print(f"{i + 1:2d}. [{case['name']} | {case['kind']}] them: {case['messages'][-1]['content']}")
    for name in ("sft_v2", "dpo_v2"):
        row = results[name][i]
        note = "   (fallback)" if row["fallback"] else ("   (filtered)" if row["raw"] != row["filtered"] else "")
        print(f"   {name}: {row['filtered']}{note}")

os.makedirs(RUN_DIR, exist_ok=True)
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump({"seed": SEED, "decoding": DECODING, "adapter_effect": ADAPTER_EFFECT, "totals": TOTALS,
               "dpo_is_a": DPO_IS_A, "votes": globals().get("VOTE_RESULT"),
               "cases": [{"name": c["name"], "kind": c["kind"], "messages": c["messages"][1:],
                          "sft_v2": results["sft_v2"][i], "dpo_v2": results["dpo_v2"][i]} for i, c in enumerate(CASES)]},
              f, indent=2, ensure_ascii=False)
print("\nSaved (private):", RESULTS_PATH)